# 01 — Project walkthrough (simple)

This notebook explains the **seizure detection** project in plain language.
It only needs the lightweight files under `docs/showcase/` — **no 244GB EEG cache and no GPU**.

> **Not a medical device.** Labels come from technician point markers, not gold-standard neurologist intervals.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

ROOT = Path("..").resolve()
SHOW = ROOT / "docs" / "showcase"
METRICS = SHOW / "metrics"
FIGS = SHOW / "figures"

headline = json.loads((METRICS / "headline.json").read_text())
audit = json.loads((METRICS / "split_audit.json").read_text())
print("Showcase root:", SHOW)
print("Best model:", headline["best_single_model"])
print("Recommended eval:", headline["recommended_eval"])

## What problem are we solving?

Hospitals record long EEG (brain-wave) sessions. Finding seizures by eye is slow.
We train a model to score **10-second windows** by how close they are to **technician workflow seizure markers** (±30 s).

That is **seizure-marker-proximity detection**, not multi-expert clinically validated seizure-interval detection.
It is also **detection** ("near a marker now?"), not reliable **forecasting** ("will a seizure happen in 30 minutes?").

Dataset: [Neurotech EEG on BDSP](https://doi.org/10.60508/v99k-ek82) (Morgan et al., 2026). This notebook is part of **Kush Patel's** ML repo — not [`bdsp-core/Neurotech-EEG-Wrangling`](https://github.com/bdsp-core/Neurotech-EEG-Wrangling).

In [ ]:
display(Image(filename=str(FIGS / "pipeline.png")))

## Data path (collection → cache)

1. **Source:** BDSP Neurotech BIDS EEG on AWS (credentialed).
2. **Select & split:** patient-level train/val/test (no patient appears in two splits).
3. **Labels:** Xltek technician CSVs → weak point markers (±30 s windows as positives).
4. **Preprocess once:** 18-channel bipolar montage, 0.5–45 Hz filter, resample to 128 Hz.
5. **Cache:** ~244 GB `window_cache/x.f32` so training does not re-read EDFs every epoch.

Full-scale run used rolling download → cache → delete raw EDF on EC2, then archived the cache to S3.

In [ ]:
prev = audit["prevalence"]
overlap = audit["patient_overlap"]
print("Patient overlap (must be 0):", overlap)
print("\nWindows / positives / prevalence")
for split, row in prev.items():
    print(
        f"  {split:5s}  n={row['n_windows']:,}  pos={row['n_pos']:,}  "
        f"prev={row['prevalence']:.2%}  patients={row['n_patients']:,}"
    )

## Model we actually shipped

| Piece | Choice |
|---|---|
| Architecture | **EEG Conformer** (~545k params) + bandpower fusion |
| Loss | Focal loss + positive class weight |
| Regularization | Channel dropout, noise, time-flip, EMA |
| Inference | Time-flip TTA + **15 s max temporal smoothing** |
| Checkpoint | `outputs/conformer_ft/best.pt` |

Resources that shaped the design: EEGNet literature, Conformer / attention-EEG papers, Flach & Kull **PRG** curves for imbalanced data, SzCORE-style event matching, and the Moutonnet et al. clinical-translation review (arXiv:2404.15332) for metrics discipline.

In [ ]:
test = headline["test_ft_max15"]
print("Headline TEST metrics (smoothed)")
print(f"  PR-AUC      {test['pr_auc']:.3f}   (95% CI {test['boot_pr_auc_95ci']})")
print(f"  ROC-AUC     {test['roc_auc']:.3f}")
print(f"  PR lift     {test['pr_lift']:.1f}× vs prevalence")
print(f"  Event sens  {test['event_sensitivity']:.0%}")
print(f"  Event FA/24h {test['event_fa_per_24h']:.1f}")

## Next notebooks

- `02_detection_results.ipynb` — recreate PR/ROC plots from saved scores
- `03_clinical_reality_check.ipynb` — operating points + honest clinical / portfolio assessment